# Phase 4 Quality Control

This notebook verifies that the trajectory extraction pipeline has generated valid trajectories, distance curves, and cosine transitions.

In [ ]:
import os
import sys
import glob
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Add root to path so we can import from src
sys.path.append(os.path.dirname(os.getcwd()))
from src.trajectories import HiddenStateTrajectory

sns.set_theme(style="whitegrid")

## 1. Load Sample Trajectories
We will scan the `data/trajectories` directory to find available models and load a few prompt trajectories from each to plot.

In [ ]:
traj_dir = "../data/trajectories"
models = [d for d in os.listdir(traj_dir) if os.path.isdir(os.path.join(traj_dir, d))] if os.path.exists(traj_dir) else []
print(f"Found models: {models}")

sample_trajectories = {}

for model in models:
    pt_files = glob.glob(os.path.join(traj_dir, model, "*.pt"))
    if not pt_files:
        continue
        
    # Load up to 3 samples per model
    sample_trajectories[model] = []
    for f in pt_files[:3]:
        traj = HiddenStateTrajectory.load(f)
        sample_trajectories[model].append(traj)

for model, trajs in sample_trajectories.items():
    print(f"\n{model} loaded {len(trajs)} samples.")
    for t in trajs:
        print(f" - Prompt {t.prompt_id} (Shape: {t.trajectory.shape})")

## 2. Distance and Cosine Similarity Curves
For each loaded model and prompt, plot the layer-to-layer transition distance and cosine similarity.

In [ ]:
if not sample_trajectories:
    print("No trajectories found to plot. Make sure to run the build_trajectories script first.")

for model, trajs in sample_trajectories.items():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Model: {model} Transition Metrics")
    
    for t in trajs:
        dist = t.layer_distance()
        cos = t.cosine_transition()
        
        layers = range(1, len(dist) + 1)
        
        ax1.plot(layers, dist, label=f"Prompt {t.prompt_id}", alpha=0.7)
        ax2.plot(layers, cos, label=f"Prompt {t.prompt_id}", alpha=0.7)
        
    ax1.set_title("Layer-to-Layer L2 Distance")
    ax1.set_xlabel("Layer Transition (L -> L+1)")
    ax1.set_ylabel("Distance")
    ax1.legend()
    
    ax2.set_title("Layer-to-Layer Cosine Similarity")
    ax2.set_xlabel("Layer Transition (L -> L+1)")
    ax2.set_ylabel("Cosine Similarity")
    ax2.legend()
    
    plt.tight_layout()
    plt.show()